<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/Demo_Transf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demo: Geometrische Transformationen
**Computer Vision – FS 2026 | Woche 4**  

Dieses Notebook zeigt die wichtigsten geometrischen Transformationen in OpenCV:  
Translation · Rotation · Skalierung · Affine Transformation · Homographie (projektive Transformation)


## 0  Imports & Hilfsfunktionen

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(images, titles, figsize=(14, 5)):
    """Zeigt eine Liste von Bildern nebeneinander an."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 1  Testbild erzeugen
Wir erzeugen ein synthetisches Testbild mit einem Schachbrettmuster – so sind Transformationen leicht erkennbar.

In [ ]:
# Synthetisches Schachbrettbild
def make_checkerboard(size=400, squares=8):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    sq = size // squares
    for r in range(squares):
        for c in range(squares):
            if (r + c) % 2 == 0:
                img[r*sq:(r+1)*sq, c*sq:(c+1)*sq] = (220, 220, 220)
            else:
                img[r*sq:(r+1)*sq, c*sq:(c+1)*sq] = (40, 40, 40)
    # Farbige Ecke zur Orientierung
    img[0:sq, 0:sq] = (0, 80, 200)
    return img

original = make_checkerboard()
h, w = original.shape[:2]
print(f'Bildgrösse: {w} x {h} Pixel')
show([original], ['Original'])


## 2  Homogene Koordinaten
Jeder 2D-Punkt (x, y) wird in homogenen Koordinaten als (x, y, 1) dargestellt.  
Das ermöglicht, **alle** Transformationen (inkl. Translation) als Matrixmultiplikation auszudrücken.

```
Affin (2×3):      Projektiv (3×3):
| a  b  tx |      | h00 h01 h02 |
| c  d  ty |      | h10 h11 h12 |
                   | h20 h21  1  |
```


In [ ]:
# Beispiel: Punkt (100, 50) in homogenen Koordinaten
p = np.array([100, 50, 1], dtype=float)

# Translation um (30, 20)
T = np.array([[1, 0, 30],
              [0, 1, 20],
              [0, 0,  1]], dtype=float)

p_trans = T @ p
print(f'Original:    {p[:2]}')
print(f'Translatiert: {p_trans[:2]}  (erwartet: [130, 70])')


## 3  Translation

In [ ]:
tx, ty = 60, 40  # Verschiebung in Pixel

M_translation = np.float32([[1, 0, tx],
                             [0, 1, ty]])

translated = cv2.warpAffine(original, M_translation, (w, h))
show([original, translated], ['Original', f'Translation (+{tx}px, +{ty}px)'])
print('Transformationsmatrix M:')
print(M_translation)


## 4  Rotation

In [ ]:
angle   = 30     # Drehwinkel in Grad
cx, cy  = w//2, h//2  # Rotationszentrum (Bildmitte)
scale   = 1.0    # Skalierungsfaktor

M_rotation = cv2.getRotationMatrix2D((cx, cy), angle, scale)
rotated = cv2.warpAffine(original, M_rotation, (w, h),
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(128, 128, 128))

show([original, rotated], ['Original', f'Rotation {angle}°'])
print('Rotationsmatrix M (2×3):')
print(np.round(M_rotation, 3))


## 5  Skalierung

In [ ]:
sx, sy = 0.7, 1.3  # Skalierungsfaktoren

M_scale = np.float32([[sx, 0,  0],
                       [ 0, sy, 0]])

scaled = cv2.warpAffine(original, M_scale, (w, h))
show([original, scaled], ['Original', f'Skalierung (sx={sx}, sy={sy})'])


## 6  Affine Transformation
Eine affine Transformation benötigt **3 Punktpaare**.  
Parallele Linien bleiben parallel – Rechtecke werden zu Parallelogrammen.


In [ ]:
# 3 Quellpunkte und Zielpunkte
src_pts = np.float32([[0,   0  ],
                       [w-1, 0  ],
                       [0,   h-1]])

dst_pts = np.float32([[50,  30 ],
                       [w-30, 20 ],
                       [20,  h-50]])

M_affine = cv2.getAffineTransform(src_pts, dst_pts)
affine = cv2.warpAffine(original, M_affine, (w, h),
                         borderMode=cv2.BORDER_CONSTANT, borderValue=(128,128,128))

show([original, affine], ['Original', 'Affine Transformation'])
print('Affine Matrix M (2×3):')
print(np.round(M_affine, 3))


## 7  Projektive Transformation (Homographie)
Eine Homographie benötigt **4 Punktpaare** und bildet eine Ebene auf eine andere ab.  
Parallele Linien bleiben **nicht** zwingend parallel – das ist die Grundlage für Image Stitching.


In [ ]:
# 4 Quellpunkte (Ecken des Bildes)
src_pts4 = np.float32([[0,   0  ],
                        [w-1, 0  ],
                        [w-1, h-1],
                        [0,   h-1]])

# Zielpunkte: trapezförmige Verzerrung (Vogelperspektive-Effekt)
dst_pts4 = np.float32([[80,  50 ],
                        [w-80, 20 ],
                        [w-40, h-30],
                        [40,  h-20]])

H, mask = cv2.findHomography(src_pts4, dst_pts4)
homography = cv2.warpPerspective(original, H, (w, h),
                                  borderMode=cv2.BORDER_CONSTANT, borderValue=(128,128,128))

show([original, homography], ['Original', 'Projektive Transformation (Homographie)'])
print('Homographie-Matrix H (3×3):')
print(np.round(H, 4))


## 8  Übersicht: Alle Transformationen


In [ ]:
show(
    [original, translated, rotated, scaled, affine, homography],
    ['Original', 'Translation', 'Rotation 30°',
     'Skalierung', 'Affin', 'Homographie'],
    figsize=(18, 4)
)


---
## Zusammenfassung
| Transformation | Funktion | Freiheitsgrade | Invarianten |
|---|---|---|---|
| Translation | `warpAffine` | 2 | Abstände, Winkel |
| Rotation | `getRotationMatrix2D` + `warpAffine` | 1 | Abstände, Winkel |
| Skalierung | `warpAffine` | 2 | Winkel, Form |
| Affin | `getAffineTransform` + `warpAffine` | 6 | Parallelität |
| Homographie | `findHomography` + `warpPerspective` | 8 | Geraden |
